In [3]:
from ase.db import connect
from ase.build import add_adsorbate

db = connect('final_optimized_db.db')
ads_db = connect('Surfaces_Li2S4.db')

# 原来写死的“局部”平面坐标（单位 Å）
local_xy = [(4,5), (6,6), (4,4), (6.5,4.3), (5,5), (6.5,4.3)]
z_list   = [4.2,   3.24,  3.24,  4.2,      6.0,   6.0    ]
symbols  = ['S',   'Li',  'Li',  'S',      'S',   'S'    ]

# 预先把这6个点的几何中心移到 (0,0)：得到“相对偏移”
mx = sum(x for x, y in local_xy) / len(local_xy)
my = sum(y for x, y in local_xy) / len(local_xy)
offset_xy = [(x - mx, y - my) for (x, y) in local_xy]

def frac_to_xy(atoms, fx, fy):
    # 把分数坐标 (fx, fy) 换成绝对 x–y（考虑非正交晶胞）
    a, b, _ = atoms.get_cell()
    X = fx * a[0] + fy * b[0]
    Y = fx * a[1] + fy * b[1]
    return (X, Y)

for row in db.select():
    atoms = row.toatoms()
    atoms_ads = atoms.copy()

    # 目标点：表面中心 (0.5, 0.5)
    cx, cy = frac_to_xy(atoms_ads, 0.5, 0.5)

    # 把每个原子放到“中心 + 相对偏移”的位置
    for (dx, dy), z, sym in zip(offset_xy, z_list, symbols):
        add_adsorbate(atoms_ads, sym, height=z, position=(cx + dx, cy + dy))

    ads_db.write(atoms_ads, model=row.formula)


In [1]:
from ase.db import connect
from ase.build import add_adsorbate

db = connect('final_optimized_db.db')
ads_db = connect('Surfaces_Li2S4_bottomright.db')

local_xy = [(4,5), (6,6), (4,4), (6.5,4.3), (5,5), (6.5,4.3)]
z_list   = [4.2,   3.24,  3.24,  4.2,      6.0,   6.0    ]
symbols  = ['S',   'Li',  'Li',  'S',      'S',   'S'    ]

mx = sum(x for x, y in local_xy) / len(local_xy)
my = sum(y for x, y in local_xy) / len(local_xy)
offset_xy = [(x - mx, y - my) for (x, y) in local_xy]

def frac_to_xy(atoms, fx, fy):
    a, b, _ = atoms.get_cell()
    return (fx * a[0] + fy * b[0], fx * a[1] + fy * b[1])

margin = 0.30  # 边距（分数坐标）
for row in db.select():
    atoms = row.toatoms()
    atoms_ads = atoms.copy()

    # 目标点：右下角（带边距）
    rx, ry = frac_to_xy(atoms_ads, 1.0 - margin, margin)

    for (dx, dy), z, sym in zip(offset_xy, z_list, symbols):
        add_adsorbate(atoms_ads, sym, height=z, position=(rx + dx, ry + dy))

    ads_db.write(atoms_ads, model=row.formula)
